# CE541E08 — Unit 3 · Day 24 — Array Manipulation: General Concepts

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 24 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | Unit conversion · SCS-CN vectorised · broadcasting · pipe sizing table |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 24"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Array Manipulation in Engineering

Today we focus on three core NumPy capabilities that make engineering computation efficient:

1. **Unit conversion on arrays** — converting entire datasets between unit systems in one line
2. **Vectorised engineering formulas** — applying SCS-CN to 6 catchments simultaneously, no loop
3. **Broadcasting** — combining arrays of different shapes to produce a matrix result (e.g. 7 diameters × 4 slopes)

These three techniques together replace most of the loops you would otherwise write for engineering parametric studies.

---
## Code Block 1 — Unit Conversion on Arrays

### What this code does

We have 5 gauging stations with measurements in mixed units — discharge in m³/s, rainfall in mm/day, elevation in feet, and area in hectares. We convert the entire dataset to SI units in one line per variable — no loop, no intermediate list.

### Why each step is taken

**`discharge * 1000` — m³/s to L/s:**
Multiplying a NumPy array by a scalar applies the multiplication to every element simultaneously. 1 m³/s = 1000 L/s.

**`rainfall / 1000` — mm to m:**
Dividing by 1000 converts millimetres to metres. Again, one line applies to all 5 values.

**`elev_ft * 0.3048` — feet to metres:**
The conversion factor 0.3048 is applied to all 5 elevation values at once.

**`area_ha / 100` — hectares to km²:**
1 km² = 100 ha. Dividing by 100 converts the entire area array.

**`rainfall_m * area_km2 * 1e6` — rainfall volume in m³:**
This multiplies two arrays element-wise (one value per station) to give the runoff volume at each station. `1e6` converts km² to m² (1 km² = 10⁶ m²). The result is one volume per station in m³.

### Algorithm

```
1. Define 5 station arrays in original units

2. Unit conversions (all vectorised — no loop):
   discharge_lps = discharge * 1000       (m³/s → L/s)
   rainfall_m    = rainfall / 1000        (mm   → m)
   elev_m        = elev_ft * 0.3048       (ft   → m)
   area_km2      = area_ha / 100          (ha   → km²)

3. Volume calculation (element-wise multiply):
   vol_m3 = rainfall_m * area_km2 * 1e6
   → one volume per station (m³)

4. Print all converted arrays
```

### Expected output

```
Discharge (L/s) : [234500.  567800.  892300. 1234500.  456700.]
Elevation (m)   : [867.1 945.7 782.4 1053.2  880.8]
Volume (m3)     : [2604960.       0. 2043432. 2393010.  209950.]
```

In [ ]:
import numpy as np

# 5 gauging stations — mixed units as received from field
discharge = np.array([234.5, 567.8, 892.3, 1234.5, 456.7])  # m³/s
rainfall  = np.array([45.6, 0, 87.3, 134.5, 22.1])          # mm/day
elev_ft   = np.array([2845, 3102, 2567, 3456, 2890])         # feet
area_ha   = np.array([1250, 890, 2340, 1780, 950])           # hectares

# Unit conversions — each applies to the entire array in one operation
discharge_lps = discharge * 1000        # m³/s → L/s  (1 m³ = 1000 L)
rainfall_m    = rainfall / 1000         # mm   → m    (1 m = 1000 mm)
elev_m        = elev_ft * 0.3048        # ft   → m    (1 ft = 0.3048 m)
area_km2      = area_ha / 100           # ha   → km²  (1 km² = 100 ha)

# Volume = depth × area; 1 km² = 1e6 m², so multiply by 1e6
vol_m3 = rainfall_m * area_km2 * 1e6   # element-wise multiply, m³

print(f"Discharge (L/s) : {discharge_lps}")
print(f"Elevation (m)   : {np.round(elev_m, 1)}")
print(f"Volume (m3)     : {np.round(vol_m3, 0)}")

### 🔁 Try this

Convert the volume from m³ to **Million Litres** (1 ML = 1000 m³).

- Add one line: `vol_ML = vol_m3 / 1000`
- Which station produces the largest runoff volume?
- Use `vol_ML.argmax() + 1` to find the station number.

---
## Code Block 2 — Vectorised SCS-CN for 6 Catchments

### What this code does

We apply the SCS-CN runoff formula to all 6 Cauvery sub-catchments simultaneously for a single design storm — no loop over catchments. Each catchment has its own area and CN value. The output is the runoff depth (mm) and volume (Mm³) for each catchment.

### Why each step is taken

**`S = 25400/CN - 254` and `Ia = 0.2*S` on arrays:**
`CN` is an array of 6 values. NumPy applies the formula element-wise — each CN gets its own S and Ia. The result is two arrays of length 6, one S per catchment and one Ia per catchment.

**`np.where(P > Ia, formula, 0.0)`:**
For each catchment, runoff only occurs if the storm depth P exceeds the initial abstraction Ia for that catchment. `np.where` applies the condition and the formula to all 6 catchments simultaneously, returning 0 where the condition is False.

**`Vol = Q/1000 * area_km2`:**
Q is in mm. Dividing by 1000 converts to metres. Multiplying by area in km² gives volume in km² × m = 10⁶ m³ = 1 Mm³ (million cubic metres). The result is one volume per catchment.

**Printing with a loop:**
Once the arrays are computed, we use a simple loop to print a formatted table — one row per catchment. The computation itself was entirely vectorised.

### Algorithm

```
1. Define 6 catchment arrays: names, area_km2, CN

2. Set storm depth P = 85 mm

3. Vectorised soil parameters:
   S  = 25400/CN - 254   → shape (6,)
   Ia = 0.2 * S          → shape (6,)

4. Vectorised SCS-CN runoff:
   Q = np.where(P > Ia,
                (P-Ia)**2 / (P-Ia+S),
                0.0)
   → shape (6,) — one Q per catchment

5. Runoff volume:
   Vol = Q/1000 * area_km2   → Mm³ per catchment

6. Print formatted table + basin total
```

### Expected output

```
Storm: 85.0 mm
Catchment       Area   CN    Q(mm)   Vol(Mm3)
-----------------------------------------------
Hemavathi       2950   72    29.91      88.230
Harangi         1930   68    22.58      43.579
Kabini          7040   75    33.64     236.826
Suvarnavathi    1575   70    26.39      41.564
Shimsha         4762   65    16.80      79.994
Arkavathi       3101   73    31.21      96.771
TOTAL          21358              :    586.964
```

In [ ]:
import numpy as np

names    = ['Hemavathi','Harangi','Kabini','Suvarnavathi','Shimsha','Arkavathi']
area_km2 = np.array([2950, 1930, 7040, 1575, 4762, 3101])
CN       = np.array([72, 68, 75, 70, 65, 73])
P        = 85.0    # design storm depth, mm

# Vectorised soil parameters — applied to all 6 CN values at once
S  = 25400 / CN - 254   # potential retention (mm), one value per catchment
Ia = 0.2 * S            # initial abstraction (mm), one value per catchment

# Vectorised SCS-CN formula — np.where replaces if-else for all 6 catchments
# Where P > Ia: apply formula; where P <= Ia: Q = 0
Q = np.where(P > Ia, (P - Ia)**2 / (P - Ia + S), 0.0)

# Runoff volume: Q mm / 1000 = Q m; Q m × area km² = Mm³
Vol = Q / 1000 * area_km2

print(f"Storm: {P} mm")
print(f"{'Catchment':<15} {'Area':>6} {'CN':>4} {'Q(mm)':>8} {'Vol(Mm3)':>10}")
print("-" * 47)
for i in range(len(names)):
    print(f"{names[i]:<15} {area_km2[i]:>6} {CN[i]:>4} {Q[i]:>8.2f} {Vol[i]:>10.3f}")
print(f"{'TOTAL':<15} {area_km2.sum():>6} {'':>4} {'':>8} {Vol.sum():>10.3f}")

### 🔁 Try this

Change the storm depth from `P = 85.0` to `P = 50.0` mm.

- Which catchments now produce zero runoff? (Ia > P)
- How does the basin total volume change?
- Which catchment is most sensitive to the change in P?

---
## Code Block 3 — Broadcasting: Anomaly Detection

### What this code does

We have 3 years of monthly rainfall data (shape `(3,12)`) and a climatological baseline (shape `(12,)`). We subtract the baseline from every year at once using **broadcasting** — NumPy automatically stretches the 1-D array to match the 2-D shape.

### Why each step is taken

**Broadcasting rule:**
When two arrays have different shapes, NumPy tries to make them compatible by stretching the smaller array. A shape `(12,)` array is broadcast against a shape `(3,12)` array by repeating the 12 values across all 3 rows. The result is a `(3,12)` anomaly matrix — one anomaly per month per year.

**`anom + clim` — actual = anomaly + climatology:**
Adding the anomaly back to the climatology recovers the actual value. This is the standard way climate datasets are structured — store the climatology once and the anomalies (which are small numbers) separately to save space.

**`(anomaly > 0).sum(axis=1)`:**
A boolean condition applied to the `(3,12)` matrix gives a `(3,12)` True/False matrix. Summing across `axis=1` (across months) gives one count per year — the number of months where rainfall was above the climatological mean.

### Algorithm

```
1. Define climatology: 12-value array (long-term monthly means)
   Define anomalies: (3,12) array — one row per year

2. actual = anom + clim
   Broadcasting: clim (12,) is stretched to (3,12)
   Result: (3,12) — actual values for each year/month

3. Print table: year rows, month columns

4. (actual > clim).sum(axis=1)
   → True/False (3,12) matrix
   → .sum(axis=1) → count of above-normal months per year
```

### Expected output

```
       J    F    M    A    M    J    J    A    S    O    N    D
2022   6   15   13   60   99  119  123  110  102   63   48   12
2023  11   10   24   48   79  154  108  121   89   83   41   15
2024   7   16   15   57   92  126  126  108   97   67   45   11
Clim   8   12   18   52   87  134  118  113   95   71   44   13
Above-normal months: [7 6 6]
```

In [ ]:
import numpy as np

# Long-term monthly climatology (mm) — 12 values
clim = np.array([8, 12, 18, 52, 87, 134, 118, 113, 95, 71, 44, 13])

# Monthly anomalies for 3 years — shape (3,12)
# Positive = wetter than normal, negative = drier than normal
anom = np.array([
    [-2,  3, -5,  8, 12, -15,  5,  -3,  7, -8,  4, -1],   # 2022
    [ 3, -2,  6, -4, -8,  20,-10,   8, -6, 12, -3,  2],   # 2023
    [-1,  4, -3,  5,  5,  -8,  8,  -5,  2, -4,  1, -2],   # 2024
])

# Broadcasting: clim has shape (12,); anom has shape (3,12)
# NumPy stretches clim to (3,12) — adding the same 12 values to each row
actual = anom + clim   # result shape: (3,12)

months = ['J','F','M','A','M','J','J','A','S','O','N','D']
print(f"{'':6} " + " ".join(f"{m:>4}" for m in months))

for y, row in enumerate(actual, 2022):
    print(f"{y:<6} " + " ".join(f"{v:>4}" for v in row))
print(f"{'Clim':<6} " + " ".join(f"{v:>4}" for v in clim))

# Count above-normal months per year
# actual > clim → (3,12) boolean; .sum(axis=1) → count per year (3,)
above = (actual > clim).sum(axis=1)
print(f"Above-normal months: {above}")

### 🔁 Try this

Compute the **standardised anomaly** for each month: `(actual - clim) / clim * 100`

This gives the percentage departure from normal.

- Which year/month combination had the largest positive anomaly?
- Use `np.unravel_index(pct_anom.argmax(), pct_anom.shape)` to find the row and column.

---
## Code Block 4 — Broadcasting: Pipe Sizing Table

### What this code does

We compute Manning's discharge for 7 pipe diameters × 4 slopes simultaneously using broadcasting — producing a `(7,4)` result matrix in a single expression. This is the full pipe sizing table used in drainage design.

### Why each step is taken

**`D_mm.reshape(-1,1)` — column vector shape `(7,1)`:**
`reshape(-1,1)` converts the 1-D diameter array into a column vector. The `-1` tells NumPy to figure out the row count automatically (7 in this case). Shape `(7,1)` means 7 rows, 1 column.

**`S.reshape(1,-1)` — row vector shape `(1,4)`:**
Similarly, the slope array becomes a row vector. Shape `(1,4)` means 1 row, 4 columns.

**Broadcasting `(7,1)` with `(1,4)` → `(7,4)`:**
NumPy broadcasts the column vector across 4 columns and the row vector across 7 rows — producing a `(7,4)` matrix where every diameter-slope combination is computed at once. No nested loop needed.

**`Q = (1/n) * R**(2/3) * Sv**0.5 * A` — fully vectorised:**
`R` and `A` are both shape `(7,1)` (depend only on diameter). `Sv` is shape `(1,4)` (depends only on slope). The result `Q` is shape `(7,4)` — one discharge for every combination.

### Algorithm

```
1. D_mm = [150,200,...,600] — shape (7,)
   S    = [0.001,...,0.005] — shape (4,)

2. Reshape:
   D = (D_mm/1000).reshape(-1,1)  → shape (7,1)
   Sv = S.reshape(1,-1)            → shape (1,4)

3. Compute per-diameter values:
   A = π(D/2)²   → shape (7,1)
   R = D/4        → shape (7,1)

4. Manning's:
   Q = (1/n) × R^(2/3) × Sv^0.5 × A
   Broadcasting: (7,1) × (1,4) → (7,4)
   Q is a (7,4) matrix of discharges

5. Print header (slope columns) + table rows (diameter rows)
```

### Expected output

```
Discharge (L/s) — Manning's full pipe (n=0.013)
D(mm) S=1:1000 S=1:500  S=1:333  S=1:200
----------------------------------------------
  150     30.6     43.3     53.0     68.5
  200     61.2     86.5    105.9    136.9
  250    106.8    151.1    185.0    239.0
  300    169.7    240.0    293.9    379.7
  375    299.8    424.0    519.2    670.7
  450    478.5    676.8    828.7   1070.6
  600    954.2   1349.4   1652.5   2134.9
```

In [ ]:
import numpy as np

D_mm = np.array([150, 200, 250, 300, 375, 450, 600])
S    = np.array([0.001, 0.002, 0.003, 0.005])
n    = 0.013

# Reshape to enable broadcasting
# D becomes a column vector (7,1) — varies down rows
# Sv becomes a row vector (1,4) — varies across columns
D  = (D_mm / 1000).reshape(-1, 1)   # shape (7,1)
Sv = S.reshape(1, -1)                # shape (1,4)

# Per-diameter quantities — shape (7,1)
A = np.pi * (D / 2)**2   # cross-sectional area, m²
R = D / 4                 # hydraulic radius, m

# Manning's — broadcasting (7,1) with (1,4) produces (7,4)
Q = (1/n) * R**(2/3) * Sv**0.5 * A   # discharge, m³/s

print(f"Discharge (L/s) — Manning's full pipe (n={n})")
# Header: slope labels
print(f"{'D(mm)':>6}", end="")
for s in S:
    print(f" S=1:{int(1/s):<4}", end="")
print()
print("-" * 46)

# One row per diameter
for i, d in enumerate(D_mm):
    print(f"{d:>6}", end="")
    for j in range(len(S)):
        print(f"  {Q[i,j]*1000:>7.1f}", end="")
    print()

### 🔁 Try this

Add a velocity check to the table. After printing each Q value, also print `V = Q/A` for that cell.

Which diameter-slope combinations give velocity **below 0.6 m/s** (silting risk)?

Use `V = (1/n) * R**(2/3) * Sv**0.5` — it is already shape `(7,4)` from broadcasting.

---
## Session Summary — Array Manipulation Concepts

| Concept | Syntax | Result |
|---|---|---|
| Scalar conversion | `arr * 1000` | All elements × 1000 |
| Element-wise multiply | `arr1 * arr2` | One product per pair |
| Vectorised formula | `np.where(P>Ia, formula, 0)` | Formula or 0 at each position |
| Reshape to column | `arr.reshape(-1,1)` | Shape `(n,1)` |
| Reshape to row | `arr.reshape(1,-1)` | Shape `(1,n)` |
| Broadcasting `(n,1)` + `(1,m)` | Automatic | Shape `(n,m)` matrix |
| Index into 2-D result | `Q[i, j]` | Value at row i, column j |
| Column volume | `(Q/1000) * area` | Runoff volume per catchment |

---
## Day 24 Assignment

8 sub-catchments with the following data:

```python
areas_ha   = np.array([125, 89, 234, 178, 95, 156, 210, 143])
CN_vals    = np.array([72, 68, 75, 70, 65, 73, 78, 69])
C_rational = np.array([0.65, 0.55, 0.70, 0.60, 0.50, 0.65, 0.72, 0.58])
P          = 78.0          # storm depth, mm
intensity  = 52.0          # mm/hr (for Rational Method)
```

1. Convert areas from ha to m² and to km²
2. Apply SCS-CN to all 8 catchments at once — compute Q (mm) for each
3. Compute Rational Method Q = C × i × A for each catchment (i in m/s, A in m²)
4. Find which catchment needs the largest drain (highest Rational Q)

### ▶ Assignment cell

In [ ]:
import numpy as np

areas_ha   = np.array([125, 89, 234, 178, 95, 156, 210, 143])
CN_vals    = np.array([72, 68, 75, 70, 65, 73, 78, 69])
C_rational = np.array([0.65, 0.55, 0.70, 0.60, 0.50, 0.65, 0.72, 0.58])
P          = 78.0
intensity  = 52.0   # mm/hr

areas_m2   = ???           # ha to m²
areas_km2  = ???           # ha to km²

# SCS-CN
S       = ???
Ia      = ???
Q_scscn = ???              # runoff depth, mm

# Rational Method: Q = C * i * A (i must be in m/s, A in m²)
i_ms       = ???           # intensity in m/s
Q_rational = ???           # peak discharge, m³/s
largest    = ???           # index of catchment with highest Q_rational

print(f"SCS-CN Q(mm)     : {np.round(Q_scscn, 2)}")
print(f"Rational Q(m3/s) : {np.round(Q_rational, 4)}")
print(f"Largest drain    : Catchment {largest + 1}")

---
- [ ] Run all cells from top to bottom — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day24.ipynb`
- [ ] Commit message: `Day 24 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*